In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix

SEED = 42

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")[train.columns]

benign_train = train[train['is_attack'] == False].reset_index(drop=True)
y_test = test['is_attack'].astype(int).values

ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
CONTEXT_PREFIX = ('source_', 'target_service_', 'target_ip_', 'flow_share_of_')

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

Xtr = prep_hard(benign_train)
Xte = prep_hard(test)[Xtr.columns]

print(f"benign_train: {len(benign_train)} | test: {len(test)} "
      f"(attack {y_test.sum()}, benign {(y_test == 0).sum()})")
print("hard features:", Xtr.shape[1])

In [ ]:
def make_detectors(seed=SEED):
    return {
        'IsolationForest': IsolationForest(n_estimators=100, contamination='auto',
                                           random_state=seed, n_jobs=-1),
        'OneClassSVM':     OneClassSVM(kernel='rbf', nu=0.05, gamma='scale'),
    }

def fit_detector(model):
    return make_pipeline(SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True),
                         StandardScaler(), model).fit(Xtr)          # benign rows only, no labels

def score(yp, name, rule):
    tn, fp, fn, tp = confusion_matrix(y_test, yp, labels=[0, 1]).ravel()
    return {'Model': name, 'Threshold': rule,
            'F1': f1_score(y_test, yp, zero_division=0),
            'BA': balanced_accuracy_score(y_test, yp),
            'FPR': fp / (fp + tn), 'Recall': tp / (tp + fn),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

PCT = 5   # flag anything below the 5th percentile of benign training scores (fixed before seeing results)

rows, fitted = [], {}
for name, m in make_detectors().items():
    pipe = fit_detector(m); fitted[name] = pipe
    s_tr, s_te = pipe.decision_function(Xtr), pipe.decision_function(Xte)   # higher = more normal
    yp = (s_te < np.percentile(s_tr, PCT)).astype(int)                       # 1 = attack
    rows.append(score(yp, name, f'benign p{PCT}'))

res = pd.DataFrame(rows)
print("===== Benign-only anomaly detection, HARD features =====")
print(res.round(3).to_string(index=False))
print("\nRefs: TRTR mean F1 0.994 / BA 0.989 | always-attack F1 0.933 / BA 0.500")

In [ ]:
rows = []
for name, pipe in fitted.items():
    s_tr, s_te = pipe.decision_function(Xtr), pipe.decision_function(Xte)
    for p in (1, 5, 10):
        rows.append(score((s_te < np.percentile(s_tr, p)).astype(int), name, f'benign p{p}'))
    rows.append(score((pipe.predict(Xte) == -1).astype(int), name, 'default'))

sens = pd.DataFrame(rows)
print("===== Threshold sensitivity =====")
print(sens[['Model', 'Threshold', 'F1', 'BA', 'FPR', 'Recall', 'FP', 'FN']].round(3).to_string(index=False))

In [ ]:
seed_rows = []
for s in (42, 45, 50):
    pipe = fit_detector(IsolationForest(n_estimators=100, contamination='auto',
                                        random_state=s, n_jobs=-1))
    s_tr, s_te = pipe.decision_function(Xtr), pipe.decision_function(Xte)
    r = score((s_te < np.percentile(s_tr, PCT)).astype(int), 'IsolationForest', f'benign p{PCT}')
    r['Seed'] = s; seed_rows.append(r)

seeds = pd.DataFrame(seed_rows)
print("===== Isolation Forest across seeds (OCSVM is deterministic) =====")
print(seeds[['Seed', 'F1', 'BA', 'FPR', 'Recall']].round(3).to_string(index=False))
print(seeds[['F1', 'BA', 'FPR']].agg(['mean', 'std']).round(3))

out = pd.concat([res.assign(Table='main'),
                 sens.assign(Table='sensitivity'),
                 seeds.assign(Table='IF_seeds')], ignore_index=True)
out.to_csv('benign_only_baseline.csv', index=False)

from google.colab import files
files.download('benign_only_baseline.csv')

In [ ]:
########### trying with my original models #####################
# ===== Cell 1: setup (same pipeline as Oct 23 BA rerun) — benign-only training =====
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, balanced_accuracy_score

SEED = 42
ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
CONTEXT_PREFIX = ('target_', 'source_', 'flow_share_')   # removed for "hard"

def to_label(s):
    return s.astype(str).str.strip().str.lower().map({'true':1,'false':0,'1':1,'0':0,'1.0':1,'0.0':0})

train = pd.read_csv('/content/train.csv', low_memory=False)
test  = pd.read_csv('/content/test.csv',  low_memory=False)
train['is_attack'] = to_label(train['is_attack']); test['is_attack'] = to_label(test['is_attack'])
test = test[train.columns]

benign_train = train[train.is_attack == 0].reset_index(drop=True)
y_test = test['is_attack'].astype(int).values

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

def make_models():
    return {'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
            'LogisticReg':  LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED),
            'SVM':          LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED)}

Xtr = prep_hard(benign_train); ytr = benign_train['is_attack'].astype(int).values
Xte = prep_hard(test)[Xtr.columns]

print('train (benign only):', Xtr.shape, '| labels in train:', np.unique(ytr))
print('test:', Xte.shape, '| attack', int(y_test.sum()), 'benign', int((y_test == 0).sum()))
assert len(benign_train) == 404 and Xtr.shape[1] == 85


In [ ]:
# ===== Cell 2: fit RF/LR/SVM on benign only, score on the same test set =====
rows = []
for name, m in make_models().items():
    pipe = make_pipeline(SimpleImputer(strategy='constant', fill_value=0), StandardScaler(), m)
    try:
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
        rows.append({'Detector': name, 'Status': 'fitted',
                     'F1': f1_score(y_test, pred, zero_division=0),
                     'BA': balanced_accuracy_score(y_test, pred),
                     'FPR': fp / (fp + tn),
                     'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                     'pred_attack': int(pred.sum())})
    except Exception as e:
        rows.append({'Detector': name, 'Status': f'ERROR: {type(e).__name__}: {str(e)[:90]}'})

res = pd.DataFrame(rows)
print(res.round(3).to_string(index=False))
res.to_csv('/content/benign_only_supervised.csv', index=False)